# PROVA NEURAL NETWORK IN SIMBIOSI CON SKLEARN

---

Leggo il nostro dict in `.pkl`

In [1]:
import pickle as pkl

with open('data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [2]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))


importo neural network

In [3]:
import sys
sys.path.append("../ML_app")

from neural_network import neural_network

---

## SMOOTHING 

provo a filtrare i dati con Savitzky-Golay

In [4]:
from scipy.signal import savgol_filter

window_size = 11
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

---

In [5]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

In [6]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

# SCALING
SCALER_OPTIONS = [MinMaxScaler()]
# PCA
N_COMPONENTS_OPTIONS = [2, 7, 19, 37, None]
# ESTIMATOR
LEARNING_RATE_OPTIONS = [0.5, 0.9, 0.95]
EPOCHE_OPTIONS = [1_000, 2_000, 10_000, 20_000]
LOSS_FUNCTION_OPTIONS = ['MSE']
ACTIVATION_FUNCTION_OPTIONS = ['sigmoide', 'tanh']

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=None)),
    
    # Step 3: Classificatore
    ("classify", neural_network(random_state=None, epoche=10_000)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per provare diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
    
},
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per saltare la riduzione
    "reduce_dim": ['passthrough'], 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente: {accuracy_finale:.4f}")

La miglior configurazione: {'classify__activation_function': 'sigmoide', 'classify__epoche': 20000, 'classify__nn_learning_rate': 0.5, 'reduce_dim__n_components': 37, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.9458
Risultato sul set indipendente: 0.9167


In [7]:
import pandas as pd
# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_pickle("results/results-nn-savgol-GridSearch.pkl")

In [8]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_scaling',
    'param_reduce_dim__n_components', 
    'param_classify__activation_function',
    'param_classify__epoche',
    'param_classify__nn_learning_rate', 
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'std_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')
renamed = analysis.rename(columns={
    'param_scaling': 'scaler',
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__activation_function': 'activation_function',
    'param_classify__epoche': 'epoche',
    'param_classify__nn_learning_rate': 'learning_rate', 
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'std_test_sensitivity': 'std_sensitivity',
    'rank_test_score': 'rank_score',
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed.head(20)

Numero totale di configurazioni provate: 144


,scaler,pca_n_components,activation_function,epoche,learning_rate,mean_score,std_score,mean_sensitivity,std_sensitivity,rank_score
48,MinMaxScaler(),37,sigmoide,20000,0.50,0.945833,0.066012,0.995833,0.032005,1
49,MinMaxScaler(),None,sigmoide,20000,0.50,0.945833,0.066012,0.995833,0.032005,1
47,MinMaxScaler(),19,sigmoide,20000,0.50,0.943750,0.070063,0.995833,0.032005,3
32,MinMaxScaler(),19,sigmoide,10000,0.50,0.943750,0.066242,0.995833,0.032005,3
34,MinMaxScaler(),None,sigmoide,10000,0.50,0.943750,0.066242,0.995833,0.032005,3
52,MinMaxScaler(),19,sigmoide,20000,0.90,0.943750,0.070063,0.995833,0.032005,3
37,MinMaxScaler(),19,sigmoide,10000,0.90,0.941667,0.070218,0.995833,0.032005,7
38,MinMaxScaler(),37,sigmoide,10000,0.90,0.941667,0.070218,0.995833,0.032005,7
19,MinMaxScaler(),None,sigmoide,2000,0.50,0.941667,0.066406,0.995833,0.032005,7
29,MinMaxScaler(),None,sigmoide,2000,0.95,0.941667,0.070218,0.995833,0.032005,7


In [ ]:
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify__activation_function,param_classify__epoche,param_classify__nn_learning_rate,param_reduce_dim__n_components,param_scaling,param_reduce_dim,...,split53_test_sensitivity,split54_test_sensitivity,split55_test_sensitivity,split56_test_sensitivity,split57_test_sensitivity,split58_test_sensitivity,split59_test_sensitivity,mean_test_sensitivity,std_test_sensitivity,rank_test_sensitivity
0,0.818308,2.063313,0.093004,0.222910,sigmoide,1000,0.50,2,MinMaxScaler(),NaN,...,0.25,0.75,0.75,1.0,0.75,0.75,0.75,0.770833,0.178487,123
1,0.005702,0.003344,0.002638,0.001261,sigmoide,1000,0.50,7,MinMaxScaler(),NaN,...,1.00,0.75,1.00,1.0,1.00,1.00,1.00,0.975000,0.087797,61
2,0.005423,0.002297,0.002151,0.001152,sigmoide,1000,0.50,19,MinMaxScaler(),NaN,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,0.995833,0.032005,25
3,0.007791,0.002935,0.003195,0.002150,sigmoide,1000,0.50,37,MinMaxScaler(),NaN,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,0.995833,0.032005,25
4,0.008295,0.002605,0.003361,0.002216,sigmoide,1000,0.50,None,MinMaxScaler(),NaN,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,0.995833,0.032005,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,0.403766,0.039051,0.003541,0.001552,tanh,10000,0.90,NaN,MinMaxScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.000000,0.000000,1
140,0.398125,0.032026,0.003193,0.001499,tanh,10000,0.95,NaN,MinMaxScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.000000,0.000000,1
141,0.807346,0.083592,0.003783,0.001566,tanh,20000,0.50,NaN,MinMaxScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.000000,0.000000,1
142,0.836776,0.064421,0.003732,0.002310,tanh,20000,0.90,NaN,MinMaxScaler(),passthrough,...,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.000000,0.000000,1
